# Intelligent Customer Support Chatbot for Real-Time Issue Resolution

## Project Overview
This notebook implements an AI-powered customer support chatbot with:
- **RAG (Retrieval Augmented Generation)** for knowledge base retrieval
- **Multiple LLM Options**: OpenAI API, Hugging Face, or template-based fallback
- **Automatic Escalation** for complex issues
- **Interactive Gradio UI**
- **Performance Evaluation** metrics

## Features:
1. Knowledge base with customer support documentation
2. Smart retrieval using semantic search
3. Context-aware response generation
4. Escalation logic for urgent issues
5. User-friendly web interface
6. Analytics and evaluation tools

## 1. Environment Setup and Installation

In [1]:
# Install required packages
!pip install -q sentence-transformers faiss-cpu langchain langchain-community
!pip install -q gradio pandas numpy matplotlib seaborn
!pip install -q openai  # Optional: for OpenAI API
!pip install -q transformers torch  # Optional: for local models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
# Import libraries
import os
import json
import re
import pandas as pd
import numpy as np
from datetime import datetime
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# For RAG
from sentence_transformers import SentenceTransformer
import faiss

# UI
import gradio as gr

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## 2. Data Preparation

Create sample customer support data and knowledge base.

In [3]:
# Create data directory
os.makedirs('data', exist_ok=True)

# Sample customer support conversations (for training/reference)
training_conversations = [
    {
        "query": "How do I reset my password?",
        "response": "To reset your password: 1) Go to the login page and click 'Forgot Password', 2) Enter your registered email address, 3) Check your email for a reset link (valid for 24 hours), 4) Click the link and create a new password with at least 8 characters. If you don't receive the email within 5 minutes, check your spam folder or contact support.",
        "category": "account"
    },
    {
        "query": "My order hasn't arrived yet",
        "response": "I understand your concern. Please provide your order number so I can track it. Standard shipping typically takes 3-5 business days, while express shipping takes 1-2 business days. You can track your order in 'My Orders' section or use the tracking link sent to your email. If it's been longer than expected, I'll escalate this immediately.",
        "category": "shipping"
    },
    {
        "query": "How can I cancel my subscription?",
        "response": "To cancel your subscription: 1) Log into your account, 2) Go to Settings > Subscription, 3) Click 'Cancel Subscription' and confirm. Your subscription remains active until the end of the current billing period. You won't be charged again. Can I help address any issues before you cancel?",
        "category": "subscription"
    },
    {
        "query": "What is your refund policy?",
        "response": "We offer a 30-day return window for most products. Items must be unused and in original packaging. To request a refund: 1) Go to My Orders, 2) Select the item, 3) Click 'Request Refund'. Refunds are processed within 5-7 business days. Digital products and personalized items are non-refundable. Original shipping costs are only refunded if the return is due to our error.",
        "category": "refund"
    },
    {
        "query": "The app keeps crashing",
        "response": "I'm sorry about the app issues. Let's troubleshoot: 1) Update to the latest version from your app store, 2) Clear the app cache in your device settings, 3) Restart your device, 4) If issues persist, uninstall and reinstall the app. Minimum requirements are iOS 13+/Android 8+, 2GB RAM. What device and OS version are you using?",
        "category": "technical"
    },
    {
        "query": "How do I update my billing information?",
        "response": "To update billing information: 1) Log into your account, 2) Go to Account Settings > Payment Methods, 3) Click 'Edit' on your current method or 'Add New Payment Method', 4) Enter new details and save. All payment information is encrypted and secure. Changes apply to future charges immediately.",
        "category": "billing"
    },
    {
        "query": "Can I change my delivery address?",
        "response": "Yes! If your order hasn't shipped yet, go to My Orders, find your order, and click 'Edit Address'. If it's already shipped, contact our support team immediately - we may be able to redirect the package or arrange redelivery depending on the carrier.",
        "category": "shipping"
    },
    {
        "query": "Do you offer student discounts?",
        "response": "Yes! We offer 20% off for verified students. To get the discount: 1) Log in or create an account, 2) Go to Discounts & Offers, 3) Click 'Verify Student Status', 4) Upload your student ID or verify with your .edu email. The discount applies automatically to eligible purchases. Re-verification required annually.",
        "category": "discount"
    },
    {
        "query": "I was charged twice for the same order",
        "response": "I sincerely apologize for this issue. Double charges are a priority concern. I'm immediately escalating this to our billing department. Please provide your order number. The duplicate charge will be reversed within 3-5 business days. You'll receive a confirmation email once resolved. Your ticket number is being generated now.",
        "category": "billing_urgent"
    },
    {
        "query": "How do I track my order?",
        "response": "To track your order: 1) Log into your account, 2) Go to My Orders, 3) Click on your order to see the tracking number and current status, 4) Click the tracking number for detailed shipping info. You should have also received a tracking email when your order shipped. Contact us if you don't see tracking info.",
        "category": "shipping"
    }
]

# Save training conversations
with open('data/training_conversations.json', 'w') as f:
    json.dump(training_conversations, f, indent=2)

print(f"✓ Created {len(training_conversations)} training conversations")

✓ Created 10 training conversations


In [4]:
# Comprehensive Knowledge Base
knowledge_base = [
    {
        "title": "Password Reset",
        "content": "To reset your password: 1) Navigate to the login page, 2) Click 'Forgot Password', 3) Enter your registered email address, 4) Check your inbox for a reset link (valid for 24 hours), 5) Click the link and create a new password. Password requirements: minimum 8 characters with at least one uppercase letter, one lowercase letter, one number, and one special character. If you don't receive the email within 5 minutes, check your spam folder. For continued issues, contact support with your account email.",
        "category": "account",
        "keywords": ["password", "reset", "forgot", "login", "access", "account"]
    },
    {
        "title": "Shipping & Delivery",
        "content": "Shipping options and timelines: Standard shipping takes 3-5 business days, Express shipping takes 1-2 business days, International orders take 7-14 business days. Free shipping on orders over $50. Tracking information is automatically sent via email once your order ships. You can also track orders in the 'My Orders' section. Delivery addresses can be modified before shipment. We support delivery to PO Boxes for standard shipping only. Signature may be required for high-value items ($500+).",
        "category": "shipping",
        "keywords": ["shipping", "delivery", "tracking", "order", "arrived", "when", "where"]
    },
    {
        "title": "Order Tracking",
        "content": "Track your orders through: 1) My Orders section in your account, 2) Tracking link in shipment confirmation email. Tracking updates occur every 24 hours. Status meanings: 'Processing' (order confirmed, preparing to ship), 'Shipped' (package in transit), 'Out for Delivery' (arriving today), 'Delivered' (completed). If tracking shows no updates for 48+ hours, contact support. Tracking numbers are provided by the shipping carrier (USPS, UPS, FedEx, DHL).",
        "category": "shipping",
        "keywords": ["track", "order", "status", "where", "package", "shipment"]
    },
    {
        "title": "Refund & Return Policy",
        "content": "Return policy: 30-day return window for most products from delivery date. Items must be unused, in original packaging with all tags attached. Non-refundable items: digital products, personalized/customized items, opened software, intimate apparel. Return process: 1) Go to My Orders, 2) Select the item, 3) Click 'Request Refund', 4) Print return label, 5) Ship item back. Refunds process within 5-7 business days after receiving returned item. Original shipping costs are non-refundable unless return is due to our error or defective product. Return shipping is customer's responsibility for non-defective items.",
        "category": "refund",
        "keywords": ["refund", "return", "money back", "cancel order", "unwanted"]
    },
    {
        "title": "Subscription Management",
        "content": "Manage subscriptions in Account Settings > Subscription. Available tiers: Basic ($9.99/month), Premium ($19.99/month), Enterprise ($49.99/month). Annual subscriptions save 20%. Cancellation: Can cancel anytime; takes effect at end of current billing period. No refunds for partial months. Upgrading: Immediate access; prorated credit applied. Downgrading: Takes effect next billing cycle. Automatic renewal can be disabled in settings. Failed payments: 3 retry attempts over 7 days, then subscription pauses.",
        "category": "subscription",
        "keywords": ["subscription", "cancel", "plan", "monthly", "billing", "renew"]
    },
    {
        "title": "Technical Support - Mobile App",
        "content": "Common app troubleshooting steps: 1) Update to latest version from app store, 2) Clear app cache in device settings (Settings > Apps > [App Name] > Clear Cache), 3) Ensure stable internet connection (WiFi or 4G/5G), 4) Restart your device, 5) Uninstall and reinstall app if problems persist. System requirements: iOS 13+ or Android 8+, 2GB RAM minimum, 100MB free storage. Supported devices: iPhone 6s and newer, Android devices from 2018 onwards. For persistent issues after troubleshooting, contact technical support with: device model, OS version, app version, and description of issue.",
        "category": "technical",
        "keywords": ["app", "crash", "bug", "error", "not working", "technical", "problem"]
    },
    {
        "title": "Payment Methods & Billing",
        "content": "Accepted payment methods: All major credit cards (Visa, Mastercard, American Express, Discover), PayPal, Apple Pay, Google Pay, Shop Pay. All transactions are encrypted with SSL/TLS. CVV verification required. Billing address must match card registration address. You can save multiple payment methods. Primary payment method is used for subscriptions and auto-renewals. Update payment info in Account Settings > Payment Methods. Payment failures trigger email notification. For billing disputes or unauthorized charges, contact billing department immediately.",
        "category": "billing",
        "keywords": ["payment", "billing", "card", "charged", "credit", "paypal", "pay"]
    },
    {
        "title": "Student Discount Program",
        "content": "Student discount: 20% off annual subscriptions and most products. Eligibility: Currently enrolled high school, college, or university students worldwide. Verification methods: 1) Valid .edu email address, 2) Student ID upload (must show name, school, and expiration date). Verification process: Account > Discounts & Offers > Verify Student Status. Discount applies automatically at checkout after verification. Re-verification required annually. Cannot be combined with other promotional offers or sale items. Discount valid while student status is active.",
        "category": "discount",
        "keywords": ["student", "discount", "education", "school", "university", "college"]
    },
    {
        "title": "Billing Issues & Disputes",
        "content": "For billing issues: Double/duplicate charges are investigated within 24 hours. Temporary authorization holds may appear as duplicate charges and clear within 3-5 business days automatically. For actual duplicate charges, we reverse them within 3-5 business days. Disputed charges: Report immediately through support ticket system. Include order number, charge amount, and date. Transaction history available in Account > Billing History. Download invoices for your records. For billing emergencies (unauthorized charges, account compromise), use priority support channel or call emergency line. Refunds process to original payment method only.",
        "category": "billing_urgent",
        "keywords": ["charged twice", "double charge", "billing error", "wrong amount", "dispute", "unauthorized"]
    },
    {
        "title": "Account Security",
        "content": "Security best practices: 1) Enable two-factor authentication (2FA) in Settings > Security, 2) Use strong, unique passwords (minimum 8 characters with mixed case, numbers, symbols), 3) Never share passwords or 2FA codes, 4) Review authorized devices regularly, 5) Log out of shared/public devices. Security features: Account lockout after 5 failed login attempts, Session timeout after 30 minutes of inactivity, Suspicious activity notifications via email, Login location tracking. If you suspect unauthorized access: 1) Change password immediately, 2) Review recent activity, 3) Revoke access from unknown devices, 4) Contact security team.",
        "category": "security",
        "keywords": ["security", "hack", "unauthorized", "2fa", "two-factor", "safe", "protect"]
    }
]

# Save knowledge base
with open('data/knowledge_base.json', 'w') as f:
    json.dump(knowledge_base, f, indent=2)

print(f"✓ Created knowledge base with {len(knowledge_base)} documents")
print(f"✓ Categories: {', '.join(set(doc['category'] for doc in knowledge_base))}")

✓ Created knowledge base with 10 documents
✓ Categories: refund, billing_urgent, account, billing, security, subscription, technical, discount, shipping


## 3. RAG System Implementation

Build a semantic search system using sentence transformers and FAISS.

In [5]:
class KnowledgeBaseRAG:
    """
    Retrieval Augmented Generation system for knowledge base
    """

    def __init__(self, knowledge_base: List[Dict], model_name: str = 'all-MiniLM-L6-v2'):
        """
        Initialize RAG system with knowledge base

        Args:
            knowledge_base: List of knowledge base documents
            model_name: Sentence transformer model name
        """
        self.knowledge_base = knowledge_base
        self.documents = []
        self.metadata = []

        # Prepare documents
        for doc in knowledge_base:
            text = f"Title: {doc['title']}\n\nContent: {doc['content']}"
            self.documents.append(text)
            self.metadata.append({
                'title': doc['title'],
                'category': doc['category'],
                'keywords': doc.get('keywords', [])
            })

        # Load embedding model
        print(f"Loading embedding model: {model_name}...")
        self.encoder = SentenceTransformer(model_name)

        # Create embeddings
        print("Creating embeddings for knowledge base...")
        self.embeddings = self.encoder.encode(self.documents, show_progress_bar=True)

        # Build FAISS index
        print("Building FAISS index...")
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(np.array(self.embeddings).astype('float32'))

        print(f"✓ RAG system initialized with {len(self.documents)} documents")

    def retrieve(self, query: str, k: int = 3) -> List[Dict]:
        """
        Retrieve top-k relevant documents for a query

        Args:
            query: User query
            k: Number of documents to retrieve

        Returns:
            List of relevant documents with metadata
        """
        # Encode query
        query_embedding = self.encoder.encode([query])

        # Search in FAISS
        distances, indices = self.index.search(np.array(query_embedding).astype('float32'), k)

        # Prepare results
        results = []
        for i, idx in enumerate(indices[0]):
            results.append({
                'document': self.documents[idx],
                'metadata': self.metadata[idx],
                'score': float(distances[0][i]),
                'rank': i + 1
            })

        return results

    def get_context(self, query: str, k: int = 2) -> str:
        """
        Get formatted context string from retrieved documents

        Args:
            query: User query
            k: Number of documents to retrieve

        Returns:
            Formatted context string
        """
        results = self.retrieve(query, k)

        context_parts = []
        for result in results:
            context_parts.append(result['document'])

        return "\n\n---\n\n".join(context_parts)

# Initialize RAG system
rag_system = KnowledgeBaseRAG(knowledge_base)

# Test retrieval
test_query = "How do I reset my password?"
results = rag_system.retrieve(test_query, k=2)

print(f"\n\nTest Query: '{test_query}'")
print(f"\nTop {len(results)} Retrieved Documents:")
for result in results:
    print(f"\n[Rank {result['rank']}] {result['metadata']['title']} (Score: {result['score']:.3f})")
    print(f"Category: {result['metadata']['category']}")
    print(f"Content Preview: {result['document'][:150]}...")

Loading embedding model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings for knowledge base...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Building FAISS index...
✓ RAG system initialized with 10 documents


Test Query: 'How do I reset my password?'

Top 2 Retrieved Documents:

[Rank 1] Password Reset (Score: 0.524)
Category: account
Content Preview: Title: Password Reset

Content: To reset your password: 1) Navigate to the login page, 2) Click 'Forgot Password', 3) Enter your registered email addr...

[Rank 2] Account Security (Score: 1.388)
Category: security
Content Preview: Title: Account Security

Content: Security best practices: 1) Enable two-factor authentication (2FA) in Settings > Security, 2) Use strong, unique pas...


## 4. Intelligent Customer Support Agent

Build the agent with escalation logic and response generation.

In [6]:
class CustomerSupportAgent:
    """
    Intelligent customer support agent with RAG and escalation
    """

    def __init__(self, rag_system: KnowledgeBaseRAG, training_data: List[Dict]):
        """
        Initialize customer support agent

        Args:
            rag_system: RAG system for knowledge retrieval
            training_data: Training conversations for reference
        """
        self.rag = rag_system
        self.training_data = training_data
        self.conversation_history = []

        # Escalation keywords (case-insensitive)
        self.escalation_keywords = [
            'urgent', 'emergency', 'immediately', 'asap',
            'manager', 'supervisor', 'speak to human', 'talk to person',
            'legal', 'lawsuit', 'attorney', 'lawyer', 'sue',
            'fraud', 'scam', 'stolen', 'unauthorized',
            'charged twice', 'double charge', 'duplicate charge',
            'never received', 'still waiting', 'weeks ago',
            'unacceptable', 'terrible', 'horrible', 'worst',
            'angry', 'furious', 'frustrated', 'disappointed'
        ]

        # Create training data index for similar query matching
        self.training_queries = [conv['query'] for conv in training_data]
        self.training_responses = [conv['response'] for conv in training_data]

        print("✓ Customer Support Agent initialized")

    def should_escalate(self, query: str) -> Tuple[bool, str]:
        """
        Determine if query should be escalated

        Args:
            query: User query

        Returns:
            Tuple of (should_escalate, reason)
        """
        query_lower = query.lower()

        # Check for escalation keywords
        for keyword in self.escalation_keywords:
            if keyword in query_lower:
                return True, f"Escalation keyword detected: '{keyword}'"

        # Check conversation length (too many turns = user frustrated)
        if len(self.conversation_history) >= 6:
            return True, "Extended conversation detected (6+ turns)"

        return False, "No escalation needed"

    def generate_response(self, query: str) -> Dict:
        """
        Generate response using RAG and template-based approach

        Args:
            query: User query

        Returns:
            Dictionary with response and metadata
        """
        # Check for escalation
        should_escalate, escalation_reason = self.should_escalate(query)

        if should_escalate:
            ticket_number = f"#{np.random.randint(10000, 99999)}"
            response = f"""I understand this is an important issue that requires immediate attention.

I'm escalating your case to our specialized support team right now. A human agent will contact you within 1 hour via email.

Your ticket number is: {ticket_number}

Please save this number for your records. Is there anything else I can help you with in the meantime?"""

            return {
                'response': response,
                'escalated': True,
                'escalation_reason': escalation_reason,
                'ticket_number': ticket_number,
                'context_used': None
            }

        # Retrieve relevant context
        retrieved_docs = self.rag.retrieve(query, k=2)
        context = self.rag.get_context(query, k=2)

        # Check for very similar training query (high similarity match)
        training_embeddings = self.rag.encoder.encode(self.training_queries)
        query_embedding = self.rag.encoder.encode([query])

        # Calculate cosine similarity
        from sklearn.metrics.pairwise import cosine_similarity
        similarities = cosine_similarity(query_embedding, training_embeddings)[0]
        best_match_idx = np.argmax(similarities)
        best_similarity = similarities[best_match_idx]

        # If very high similarity (>0.7), use training response as template
        if best_similarity > 0.7:
            base_response = self.training_responses[best_match_idx]
        else:
            # Generate response from retrieved context
            base_response = self._generate_from_context(query, retrieved_docs)

        return {
            'response': base_response,
            'escalated': False,
            'escalation_reason': None,
            'ticket_number': None,
            'context_used': context,
            'retrieved_docs': [doc['metadata']['title'] for doc in retrieved_docs],
            'similarity_score': float(best_similarity)
        }

    def _generate_from_context(self, query: str, retrieved_docs: List[Dict]) -> str:
        """
        Generate response from retrieved context (template-based)
        """
        if not retrieved_docs:
            return "I apologize, but I don't have specific information about that. Let me escalate this to a human agent who can better assist you."

        # Extract key information from top document
        top_doc = retrieved_docs[0]
        content = top_doc['document']

        # Parse content (remove title prefix)
        if 'Content:' in content:
            main_content = content.split('Content:')[1].strip()
        else:
            main_content = content

        # Create response template
        response = f"""I'd be happy to help you with that!

{main_content}

Is there anything else you'd like to know?"""

        return response

    def chat(self, query: str) -> Dict:
        """
        Main chat interface

        Args:
            query: User query

        Returns:
            Response dictionary
        """
        # Generate response
        result = self.generate_response(query)

        # Add to conversation history
        self.conversation_history.append({
            'query': query,
            'response': result['response'],
            'escalated': result['escalated'],
            'timestamp': datetime.now().isoformat()
        })

        return result

    def reset_conversation(self):
        """Reset conversation history"""
        self.conversation_history = []
        print("✓ Conversation reset")

# Initialize agent
agent = CustomerSupportAgent(rag_system, training_conversations)

# Test the agent
print("\n" + "="*80)
print("TESTING CUSTOMER SUPPORT AGENT")
print("="*80)

test_queries = [
    "How do I reset my password?",
    "I was charged twice for my order!",
    "What's your refund policy?"
]

for query in test_queries:
    print(f"\n\nUser: {query}")
    result = agent.chat(query)
    print(f"\nAgent: {result['response'][:300]}...")
    print(f"\nMetadata:")
    print(f"  - Escalated: {result['escalated']}")
    if result.get('retrieved_docs'):
        print(f"  - Retrieved Docs: {', '.join(result['retrieved_docs'])}")
    if result.get('similarity_score'):
        print(f"  - Similarity Score: {result['similarity_score']:.3f}")
    print("-" * 80)

    # Reset for next test
    agent.reset_conversation()

✓ Customer Support Agent initialized

TESTING CUSTOMER SUPPORT AGENT


User: How do I reset my password?

Agent: To reset your password: 1) Go to the login page and click 'Forgot Password', 2) Enter your registered email address, 3) Check your email for a reset link (valid for 24 hours), 4) Click the link and create a new password with at least 8 characters. If you don't receive the email within 5 minutes, che...

Metadata:
  - Escalated: False
  - Retrieved Docs: Password Reset, Account Security
  - Similarity Score: 1.000
--------------------------------------------------------------------------------
✓ Conversation reset


User: I was charged twice for my order!

Agent: I understand this is an important issue that requires immediate attention. 

I'm escalating your case to our specialized support team right now. A human agent will contact you within 1 hour via email.

Your ticket number is: #91545

Please save this number for your records. Is there anything else I ...

Metadata:
  -

## 5. Gradio User Interface

Create an interactive web interface for the chatbot.

In [7]:
# Create a fresh agent instance for the UI
ui_agent = CustomerSupportAgent(rag_system, training_conversations)

def chatbot_interface(message, history):
    """
    Gradio chatbot function

    Args:
        message: User message
        history: Chat history (list of [user_msg, bot_msg] pairs)

    Returns:
        Bot response string
    """
    try:
        # Get response from agent
        result = ui_agent.chat(message)

        response = result['response']

        # Add escalation badge if escalated
        if result['escalated']:
            response = "🚨 **ESCALATED TO HUMAN SUPPORT** 🚨\n\n" + response

        # Add metadata footer (optional - comment out if too verbose)
        if result.get('retrieved_docs') and not result['escalated']:
            footer = f"\n\n---\n*📚 Sources: {', '.join(result['retrieved_docs'])}*"
            response += footer

        return response

    except Exception as e:
        return f"I apologize, but I encountered an error: {str(e)}. Please try rephrasing your question or contact support directly."

def reset_chat():
    """
    Reset conversation when clear button is clicked
    """
    ui_agent.reset_conversation()
    return []

# Create Gradio interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🤖 Intelligent Customer Support Chatbot

        Welcome! I'm your AI customer support assistant. I can help you with:
        - 🔐 Account management (passwords, billing, security)
        - 📦 Order tracking and shipping questions
        - 💰 Refunds and returns
        - 📱 Technical support for our mobile app
        - 💳 Payment and subscription issues
        - 🎓 Student discounts and promotions

        If I can't resolve your issue, I'll escalate it to our human support team immediately.
        """
    )

    chatbot = gr.Chatbot(
        height=500,
        bubble_full_width=False,
        avatar_images=(None, "🤖")
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type your question here... (e.g., 'How do I reset my password?')",
            scale=4,
            container=False
        )
        submit = gr.Button("Send", scale=1, variant="primary")

    with gr.Row():
        clear = gr.Button("🗑️ Clear Chat")

    gr.Examples(
        examples=[
            "How do I reset my password?",
            "Where is my order?",
            "What's your refund policy?",
            "The app keeps crashing on my phone",
            "Do you offer student discounts?",
            "How do I cancel my subscription?",
            "I was charged twice for my order",
            "Can I change my delivery address?",
        ],
        inputs=msg,
        label="Try these example questions:"
    )

    # Event handlers
    def respond(message, chat_history):
        bot_message = chatbot_interface(message, chat_history)
        chat_history.append((message, bot_message))
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit.click(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: (ui_agent.reset_conversation(), []), None, [chatbot], queue=False)

    gr.Markdown(
        """
        ---
        ### 💡 Tips:
        - Be specific with your questions for better responses
        - Mention order numbers, account emails, or other relevant details
        - For urgent issues, use keywords like "urgent" or "emergency" for faster escalation

        ### 🔧 Technical Details:
        - **RAG System**: Uses semantic search with sentence transformers
        - **Knowledge Base**: 10+ comprehensive support documents
        - **Auto-Escalation**: Intelligent detection of urgent/complex issues
        """
    )

print("✓ Gradio interface created!")
print("\nLaunching chatbot interface...")

✓ Customer Support Agent initialized
✓ Gradio interface created!

Launching chatbot interface...


In [8]:
# Launch the interface
demo.launch(
    share=False,  # Set to True to create a public link
    server_port=7860,
    server_name="0.0.0.0",  # Allow external access
    show_error=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>